# Tree-ensemble training and interpretation

Purpose

Train tree ensemble → evaluate → SHAP → interpret biology.

In [ ]:
X = pd.read_parquet("data/processed/synthetic_exerkinemap.parquet")
y = X.pop("response")

## Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Train Ensemble

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import numpy as np

models = [
    RandomForestClassifier(),
    ExtraTreesClassifier(),
    XGBClassifier(eval_metric="logloss")
]

for m in models:
    m.fit(X_train, y_train)

preds = np.mean(
    [m.predict_proba(X_test)[:,1] for m in models],
    axis=0
)

 ## Evaluation

from sklearn.metrics import roc_auc_score

roc_auc_score(y_test, preds)

## SHAP Interpretation

In [ ]:
import shap

explainer = shap.TreeExplainer(models[0])
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

Now you can see which exerkine-derived features drive response.